## Task 3, part 4 NMF + KNN model for RNA fingerprint prediction

This model predicts each held-out perturbation's RNA log2FC fingerprint by (1) compressing the training fingerprints into a small set of non-negative "gene programs" via NMF, and (2) estimating a new perturbation's program usage as the average usage of its k nearest neighbours in gene-identity space, where neighbours are found using signature-based gene embeddings.

In [7]:
import numpy as np                                   
import pandas as pd                                   
from scipy.stats import pearsonr, spearmanr           
from sklearn.decomposition import PCA, NMF            
from sklearn.preprocessing import StandardScaler       
from sklearn.neighbors import NearestNeighbors         

SEED = 0                                               
np.random.seed(SEED)

In [8]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")   
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()  
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()    

gene_cols = pert_FC_selected.columns.tolist()          
selected_50 = train_40 + test_10                       
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()  

pert_FC_selected.shape

(150, 2042)

## Build gene embeddings

Every gene in gene_cols gets a compact "identity vector" describing how it tends to respond across the 40 training perturbations. We take the training block of pert_FC_selected, transpose it so genes become rows (samples) and perturbation×condition combinations become features, z-score, then PCA down to 32 components. This embedding is what lets us later find, for an unseen perturbation, which training perturbations targeted the most similar genes -- it never sees the test perturbations.


In [9]:
EMB_DIM = 32   # number of PCA components used as each gene's identity vector

def build_gene_embeddings(fc_df, train_perts, gene_cols, n_components=EMB_DIM):
    """Embed every gene using its own column of log2FC values, restricted to training perturbations."""
    train_block = fc_df.loc[train_perts]              # (n_train_rows, n_genes), only training perturbation rows
    gene_signatures = train_block.T.values             # transpose -> (n_genes, n_train_rows), genes as samples

    scaler = StandardScaler()                          # z-score each training-row feature across genes
    gene_signatures = scaler.fit_transform(gene_signatures)

    n_components = min(n_components, gene_signatures.shape[0], gene_signatures.shape[1])  # can't exceed matrix rank
    pca = PCA(n_components=n_components, random_state=SEED)   # reduce each gene's signature to n_components numbers
    embeddings = pca.fit_transform(gene_signatures)     # (n_genes, n_components)

    return pd.DataFrame(embeddings, index=gene_cols)    # one embedding row per gene, indexed by gene name

def get_gene_embedding(gene, emb_df, dim):
    """Look up a perturbation's targeted gene's embedding; fall back to zeros if the gene wasn't a measured column."""
    if gene in emb_df.index:
        return emb_df.loc[gene].values.astype(np.float32)
    return np.zeros(dim, dtype=np.float32)              # perturbed gene not among the selected columns

gene_embeddings = build_gene_embeddings(pert_FC_selected, train_40, gene_cols)   # fit only on train_40
gene_embeddings.shape

(2042, 32)

## Reduce the target: NMF on the training RNA FC vectors

The RNA fingerprint has 2042 signed log2FC values, but NMF requires non-negative input. We split each fingerprint into an "up" block (positive values, rest zeroed) and a "down" block (absolute value of negative values, rest zeroed) and stack them side by side, giving a (n_rows, 4084) non-negative matrix. NMF on this matrix yields a small set of non-negative "programs" (H) and, for every training perturbation/condition row, a non-negative usage vector (W) over those programs. Splitting is fully reversible: subtracting the down block from the up block recovers the original signed fingerprint exactly.

In [10]:
N_FACTORS = 10          # number of NMF programs, kept small given only 40 training perturbations
n_genes = len(gene_cols)

def to_nonneg(Y):
    """Split a signed log2FC matrix into a non-negative [up | down] matrix."""
    return np.hstack([np.maximum(Y, 0), np.maximum(-Y, 0)])   # up block, down block, glued side by side

def from_nonneg(Y_nn, n_genes):
    """Undo to_nonneg: recover the signed log2FC matrix from the [up | down] representation."""
    return Y_nn[:, :n_genes] - Y_nn[:, n_genes:]               # up minus down = original signed values

Y_train_full = pert_FC_selected.loc[train_40].values            
meta_train = list(pert_FC_selected.loc[train_40].index)         

Y_train_nn = to_nonneg(Y_train_full)                             # (120, 4084), guaranteed non-negative

nmf = NMF(n_components=N_FACTORS, init="nndsvda", random_state=SEED, max_iter=1000)  # deterministic init, more iterations for stability
W_train = nmf.fit_transform(Y_train_nn)    # (120, 10) -- program usage per training row
H = nmf.components_                         # (10, 4084) -- the fixed programs themselves

recon_err = np.mean((Y_train_nn - W_train @ H) ** 2)   # in-sample reconstruction error, sanity check only
print(f"NMF reconstruction MSE (train_40): {recon_err:.4f}")

NMF reconstruction MSE (train_40): 0.0004


## KNN prediction of program usage

Since a gene's embedding is condition-independent, we first average each training perturbation's program-usage vector (W) across its 3 conditions. To predict a new perturbation's program usage, we find the k training perturbations whose gene embeddings are closest to the query's, and average their program-usage vectors. The exclude_self option lets the same function be reused for leave-one-out cross-validation, where a training perturbation must not be allowed to be its own neighbour.

In [11]:
# average program usage per perturbation across its 3 conditions
W_per_pert = {}
for pert in train_40:                                              
    rows = [i for i, (p, c) in enumerate(meta_train) if p == pert]  
    W_per_pert[pert] = W_train[rows].mean(axis=0)                   

def knn_predict_W(query_pert, pool_perts, k, exclude_self=True):
    """Predict a perturbation's program-usage vector as the mean of its k nearest neighbours' vectors."""
    fit_pool = [p for p in pool_perts if not (exclude_self and p == query_pert)]  # remove query from its own neighbour pool if requested

    pool_emb = np.stack([get_gene_embedding(p, gene_embeddings, EMB_DIM) for p in fit_pool])  # embeddings of the pool
    query_emb = get_gene_embedding(query_pert, gene_embeddings, EMB_DIM).reshape(1, -1)        # embedding of the query, as a 1-row matrix

    nn = NearestNeighbors(n_neighbors=min(k, len(fit_pool)))   # cap k at pool size to avoid errors
    nn.fit(pool_emb)
    _, idx = nn.kneighbors(query_emb)                          # indices of the k nearest pool perturbations

    neighbor_perts = [fit_pool[j] for j in idx[0]]             # names of the k nearest neighbours
    W_pred = np.mean([W_per_pert[p] for p in neighbor_perts], axis=0)  # plain average of their program usage
    return W_pred

def predict_fingerprint(query_pert, k, exclude_self, pool_perts=train_40):
    """Predict the full signed log2FC fingerprint for a perturbation, independent of condition."""
    W_pred = knn_predict_W(query_pert, pool_perts, k, exclude_self)   # predicted program usage
    Y_nn_pred = W_pred @ H                                             # (4084,) reconstruction in [up | down] space
    return from_nonneg(Y_nn_pred[None, :], n_genes)[0]                 # back to a signed (2042,) fingerprint

## Choosing k via leave-one-out cross-validation

Hold out one training perturbation at a time, predict its fingerprint from the other 39 (per condition), and compare a few candidate k values by MSE. Note: the prediction is condition-independent (it depends only on gene identity), so the same predicted fingerprint is compared against every condition's true value for that perturbation, mirroring how the underlying model works.

In [12]:
candidate_ks = [1, 2, 3, 5, 8, 12, 20]        

cv_mse_by_k = {}                               
for k in candidate_ks:                         
    squared_errors = []                        
    for cond in conditions:                    
        for gene in train_40:                  
            # predict_fingerprint excludes the query gene itself from the pool, so this is a leave-one-out prediction
            pred = predict_fingerprint(gene, k, exclude_self=True)      # predict the left-out perturbation's fingerprint
            true = pert_FC_selected.loc[(gene, cond)].values            # extract the actual left-out fingerprint
            squared_errors.append(np.mean((true - pred) ** 2))          # append the squared error to the list

    cv_mse_by_k[k] = np.mean(squared_errors)   # average MSE for this k, over 120 = 3 x 40 entries

best_k = min(cv_mse_by_k, key=cv_mse_by_k.get)  # k with the lowest cross-validated MSE
cv_mse_by_k, best_k

({1: np.float32(0.0028165267),
  2: np.float32(0.0024731492),
  3: np.float32(0.0024070665),
  5: np.float32(0.0023764262),
  8: np.float32(0.002352363),
  12: np.float32(0.0023387235),
  20: np.float32(0.0023517748)},
 12)

## Predict test perturbation effects on the RNA expression FC vectors and evaluate the model

Use the chosen k to predict each of the 10 held-out perturbations' RNA fingerprint from their gene identity, then evaluate with the same metrics used for the baseline and ridge models so results are directly comparable.

In [13]:
# for each test perturbation predict the log2FC vector under each condition using the k found in the previous part
nmf_predictions = {
    (gene, cond): predict_fingerprint(gene, best_k, exclude_self=False)  # test genes are never in the training pool, so no exclusion needed
    for cond in conditions        # iterate over all three conditions
    for gene in test_10           # iterate over all 10 test perturbations
}

def evaluate_predictions(true_df, predictions_by_row):
    """Compare true and predicted fingerprints row by row and return a tidy per-row metrics DataFrame."""
    records = []                                            
    for (pert, cond), true_fc in true_df.iterrows():        
        pred_fc = predictions_by_row[(pert, cond)]          
        pearson_r, _ = pearsonr(true_fc, pred_fc)           
        spearman_r, _ = spearmanr(true_fc, pred_fc)         
        mse = np.mean((true_fc - pred_fc) ** 2)             
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        })
    return pd.DataFrame(records)                              # return as a DataFrame


nmf_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], nmf_predictions)  # run evaluation on the test set
nmf_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.822119,0.392718,0.001600
1,KCNN4,IFNγ,0.851150,0.334713,0.001013
2,KCNN4,Co-culture,0.861680,0.332121,0.000989
3,TIMM50,Control,0.746076,0.343015,0.002788
4,TIMM50,IFNγ,0.633092,0.263206,0.003111
5,TIMM50,Co-culture,0.644383,0.138666,0.004066
6,TXNDC17,Control,0.791833,0.473999,0.004961
7,TXNDC17,IFNγ,0.589691,0.368122,0.004576
8,TXNDC17,Co-culture,0.768206,0.397075,0.003947
9,CORO1A,Control,0.798199,0.307694,0.001086


In [14]:
metrics = ["pearson_r", "spearman_r", "mse"]   # relevant metrics to extract

# model performance in the metrics, averaged for each condition
per_condition = nmf_eval.groupby("condition")[metrics].agg(["mean", "std"])

# model performance in the metrics but averaged across all test perturbations and conditions
overall = nmf_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.571218  0.494608   0.240577  0.128415  0.004290  0.006654
Control     0.757694  0.118090   0.349190  0.116828  0.002490  0.001457
IFNγ        0.717292  0.253933   0.311369  0.096261  0.002363  0.001708

In [15]:
overall

,pearson_r,spearman_r,mse
mean,0.682068,0.300379,0.003048
std,0.326953,0.119690,0.004014


## Discussion

The model uses each perturbation's 32-dimensional gene-identity embedding, derived via PCA on the transposed training RNA fingerprints, to look up its k nearest training perturbations and average their NMF program-usage vectors. NMF was fit once on all 40 training perturbations across all three conditions, factorizing the non-negative up/down-split fingerprints into 10 additive gene programs. Since the gene embedding is condition-independent, the model predicts a single fingerprint per perturbation, compared against the true fingerprint under each condition. k was chosen via leave-one-out cross-validation across the 40 training perturbations.

With an overall Pearson correlation of 0.682 (baseline: 0.688), a Spearman correlation of 0.300 (baseline: 0.349), and an MSE of 0.00305 (baseline: 0.00297), the model performs marginally worse than the mean baseline across all three metrics, most noticeably in the rank-based Spearman correlation. This suggests the added structure (NMF programs plus KNN) is not capturing perturbation-specific signal beyond what a simple condition-wise mean already provides. The high standard deviation of the Pearson correlation (0.327) relative to its mean also points to substantial variability across perturbations, with some reconstructed reasonably well and others essentially uncorrelated with the truth.

This is likely due to the architecture. With only 39 candidate neighbours at prediction time, the neighbourhoods are neither large enough to average out noise nor small enough to isolate a truly similar subset of perturbations. In a 32-dimensional embedding space populated by so few points, nearest neighbours may just be the least distant among a sparse set rather than meaningfully similar biologically. Additionally, because the KNN step is condition-independent, the model structurally cannot capture any condition-specific component of a perturbation's effect, while the baseline is condition-specific by construction.

Finally, fitting NMF globally across all three conditions rather than per condition (as done for the target PCA in the ridge model) gives NMF more rows to work with but likely blurs condition-specific program structure. A per-condition factorization, or a condition-aware neighbour search, are plausible directions for improvement.